In [142]:
import polars as pl
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from linearmodels.iv import compare
from linearmodels.panel.results import compare
import statsmodels.api as sm
from statsmodels.api import qqplot
from linearmodels.panel import PanelOLS
from scipy import stats as scipy_stats

import pyreadstat as prs
import pygwalker as pyg

import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt

import json

from tqdm.notebook import tqdm
from itables import init_notebook_mode

alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

# add variables

In [143]:
additional_health_variables = pl.read_excel('../data/disease_variables.xlsx')
additional_health_variables

var,label,fill
str,str,str
"""m20_61""","""chronic_heart""","""no"""
"""m20_62""","""chronic_lung""","""no"""
"""m20_63""","""chronic_liver""","""no"""
"""m20_64""","""chronic_kidney""","""no"""
"""m20_65""","""chronic_stomach""","""no"""
"""m20_66""","""chronic_spinal""","""no"""
"""m20_612""","""disease_neuro""","""no"""
"""m20_613""","""disease_eye""","""no"""
"""m20_615""","""allegies""","""no"""


In [144]:
cols_disease = additional_health_variables['var'].to_list()
cols_disease

['m20_61',
 'm20_62',
 'm20_63',
 'm20_64',
 'm20_65',
 'm20_66',
 'm20_612',
 'm20_613',
 'm20_615',
 'm20_8']

In [145]:
cols = ['idind', 'year', 'h7_2', 'h7_1', 'origsm', 'region', 'psu', 'status', 'age', 'educ', 'marst', 'h5', 'm20_7', 'j1', 'j60', 'j40', 'j363']

In [146]:
cols + cols_disease

['idind',
 'year',
 'h7_2',
 'h7_1',
 'origsm',
 'region',
 'psu',
 'status',
 'age',
 'educ',
 'marst',
 'h5',
 'm20_7',
 'j1',
 'j60',
 'j40',
 'j363',
 'm20_61',
 'm20_62',
 'm20_63',
 'm20_64',
 'm20_65',
 'm20_66',
 'm20_612',
 'm20_613',
 'm20_615',
 'm20_8']

In [147]:
education_levels = pl.read_excel('../data/all_education_levels.xlsx', sheet_name='selected_levels')
education_levels

educ,educ_level
str,str
"""1-2 years in Institute, Univer…","""university"""
"""7 grades of school""","""school_or_less"""
"""6 grades of school""","""school_or_less"""
"""3 grades of school""","""school_or_less"""
"""7-9 grades of school [unfinish…","""common"""
…,…
"""7-9 grades of school [unfinish…","""common"""
"""Graduate school, residency wit…","""common"""
"""10 and more grades of school &…","""higher"""


In [148]:
marital_status = pl.read_excel('../data/all_marst_levels.xlsx', sheet_name='selected_levels')
marital_status

marst,is_married
str,i64
"""In a registered marriage""",1
"""Divorsed and not remarried""",0
"""Living together, not registere…",1
"""Registered, not living togethe…",1
"""Never married""",0
"""Widower or widow""",0


In [149]:
period_relevance = pl.read_excel('../data/period_relevance.xlsx').with_columns(
    pl.col('year').cast(pl.Float64)
)
period_relevance

year,period_relevance
f64,i64
2014.0,-5
2015.0,-4
2016.0,-3
2017.0,-2
2018.0,-1
…,…
2020.0,1
2021.0,2
2022.0,3


In [150]:
is_town = pl.read_excel('../data/is_town.xlsx')
is_town

status,is_town
str,i64
"""town""",1
"""oblastnoy center""",1
"""PGT""",1
"""rural""",0


In [151]:
panel = pl.read_parquet('../data/RLMS_IND_2015_2024_eng_prepared.parquet', columns=cols + cols_disease)
for var, label, fill in zip(additional_health_variables['var'], additional_health_variables['label'], additional_health_variables['fill']):
    panel = panel.with_columns(
        pl.col(var).fill_null(fill).alias(label)
    )
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
34896.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",66.0,"""Institute, University, Academy…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",44000.0,null,14000.0,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no"""
34897.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",62.5,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14000.0,null,13200.0,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no"""
34907.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",54.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""Yes""","""You are not working""",7000.0,null,7000.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group"""
34908.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",58.5,"""Technical, medical, musis etc …","""In a registered marriage""","""MALE""","""No""","""You are currently working""",7000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no"""
34919.0,2014.0,"""January""",19.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",48.5,"""10 and more grades of school &…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",4500.0,null,null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""","""no"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59407.0,2024.0,"""November""",27.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",8.0,null,null,"""FEMALE""","""No""",null,null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no"""
59408.0,2024.0,"""October""",9.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",73.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",31000.0,null,31000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no"""
59409.0,2024.0,"""October""",11.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",53.0,"""10 and more grades of school &…","""Divorsed and not remarried""","""MALE""","""No""","""You are not working""",0.0,null,null,"""No""","""No""","""No""","""No""","""No""","""Y

In [152]:
panel = panel.with_columns(
    date = pl.concat_str([pl.col('year').cast(pl.Int32), pl.col('h7_2'), pl.col('h7_1').cast(pl.Int32)], separator=' ').str.strptime(pl.Date, format='%Y %B %d')
)
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,date
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date
34896.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",66.0,"""Institute, University, Academy…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",44000.0,null,14000.0,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05
34897.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",62.5,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14000.0,null,13200.0,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05
34907.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",54.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""Yes""","""You are not working""",7000.0,null,7000.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""",2014-01-18
34908.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",58.5,"""Technical, medical, musis etc …","""In a registered marriage""","""MALE""","""No""","""You are currently working""",7000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-18
34919.0,2014.0,"""January""",19.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",48.5,"""10 and more grades of school &…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",4500.0,null,null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""","""no""",2014-01-19
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59407.0,2024.0,"""November""",27.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",8.0,null,null,"""FEMALE""","""No""",null,null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2024-11-27
59408.0,2024.0,"""October""",9.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",73.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",31000.0,null,31000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",2024-10-09
59409.0,2024.0,"""October""",11.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",53.0,"""10 and more grades of school &…","""Divorsed and not remarried""","""MALE""","""No""",

In [153]:
panel = panel.with_columns(
    is_employed = pl.when(pl.col('j1').is_in(['You are currently working', 'You are on paid leave: maternity leave or taking care of a child under 3 years o', 'You are on another kind of paid leave'])).then(1).otherwise(0)
)
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,date,is_employed
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,i32
34896.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",66.0,"""Institute, University, Academy…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",44000.0,null,14000.0,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,1
34897.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",62.5,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14000.0,null,13200.0,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,0
34907.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",54.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""Yes""","""You are not working""",7000.0,null,7000.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""",2014-01-18,0
34908.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",58.5,"""Technical, medical, musis etc …","""In a registered marriage""","""MALE""","""No""","""You are currently working""",7000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-18,1
34919.0,2014.0,"""January""",19.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",48.5,"""10 and more grades of school &…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",4500.0,null,null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""","""no""",2014-01-19,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59407.0,2024.0,"""November""",27.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",8.0,null,null,"""FEMALE""","""No""",null,null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2024-11-27,0
59408.0,2024.0,"""October""",9.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",73.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",31000.0,null,31000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",2024-10-09,0
59409.0,2024.0,"""October""",11.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",53.0,"""10 and more grades of school &…","""Divorsed and not r

In [154]:
panel = panel.with_columns(
    has_disablity = pl.when(pl.col('m20_7').is_in(['No', 'NO ANSWER'])).then(0).otherwise(1)
)
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,date,is_employed,has_disablity
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,i32,i32
34896.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",66.0,"""Institute, University, Academy…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",44000.0,null,14000.0,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,1,0
34897.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",62.5,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14000.0,null,13200.0,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,0,0
34907.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",54.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""Yes""","""You are not working""",7000.0,null,7000.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""",2014-01-18,0,1
34908.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",58.5,"""Technical, medical, musis etc …","""In a registered marriage""","""MALE""","""No""","""You are currently working""",7000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-18,1,0
34919.0,2014.0,"""January""",19.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",48.5,"""10 and more grades of school &…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",4500.0,null,null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""","""no""",2014-01-19,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59407.0,2024.0,"""November""",27.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",8.0,null,null,"""FEMALE""","""No""",null,null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2024-11-27,0,0
59408.0,2024.0,"""October""",9.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",73.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",31000.0,null,31000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",2024-10-09,0,0
59409.0,2024.0,"""October""",11.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",53.0,"""10 and more grades o

In [155]:
panel = panel.with_columns(
    is_employed_has_disablity = pl.col('is_employed') * pl.col('has_disablity')
)
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,date,is_employed,has_disablity,is_employed_has_disablity
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,i32,i32,i32
34896.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",66.0,"""Institute, University, Academy…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",44000.0,null,14000.0,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,1,0,0
34897.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",62.5,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14000.0,null,13200.0,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,0,0,0
34907.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",54.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""Yes""","""You are not working""",7000.0,null,7000.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""",2014-01-18,0,1,0
34908.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",58.5,"""Technical, medical, musis etc …","""In a registered marriage""","""MALE""","""No""","""You are currently working""",7000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-18,1,0,0
34919.0,2014.0,"""January""",19.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",48.5,"""10 and more grades of school &…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",4500.0,null,null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""","""no""",2014-01-19,1,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59407.0,2024.0,"""November""",27.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",8.0,null,null,"""FEMALE""","""No""",null,null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2024-11-27,0,0,0
59408.0,2024.0,"""October""",9.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",73.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",31000.0,null,31000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",2024-10-09,0,0,0
59409.0,2024.0,"""October""",11.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Ob

In [156]:
panel = panel.with_columns(
    is_employed_no_disablity = pl.col('is_employed') * pl.col('has_disablity').replace({0:1, 1:0})
)
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,date,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,i32,i32,i32,i32
34896.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",66.0,"""Institute, University, Academy…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",44000.0,null,14000.0,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,1,0,0,1
34897.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",62.5,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14000.0,null,13200.0,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,0,0,0,0
34907.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",54.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""Yes""","""You are not working""",7000.0,null,7000.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""",2014-01-18,0,1,0,0
34908.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",58.5,"""Technical, medical, musis etc …","""In a registered marriage""","""MALE""","""No""","""You are currently working""",7000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-18,1,0,0,1
34919.0,2014.0,"""January""",19.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",48.5,"""10 and more grades of school &…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",4500.0,null,null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""","""no""",2014-01-19,1,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59407.0,2024.0,"""November""",27.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",8.0,null,null,"""FEMALE""","""No""",null,null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2024-11-27,0,0,0,0
59408.0,2024.0,"""October""",9.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",73.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",31000.0,null,31000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",2024-10-09,0,0,0,0
59409.0,2024.0,"""October""",11.0,"""Yes, it is an adress from

In [157]:
panel = panel.with_columns(
    net_job = pl.col('j60').fill_null(0) - pl.col('j363').fill_null(0)
)
panel = panel.with_columns(
    pl.when(pl.col('net_job') < 0).then(0).otherwise('net_job').alias('net_job')
)
# panel = panel.with_columns(
#     pl.when(pl.col('is_employed') == 0).then(0).otherwise('net_job').alias('net_job')
# ) 
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,date,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,i32,i32,i32,i32,f64
34896.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",66.0,"""Institute, University, Academy…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",44000.0,null,14000.0,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,1,0,0,1,30000.0
34897.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",62.5,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14000.0,null,13200.0,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,0,0,0,0,800.0
34907.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",54.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""Yes""","""You are not working""",7000.0,null,7000.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""",2014-01-18,0,1,0,0,0.0
34908.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",58.5,"""Technical, medical, musis etc …","""In a registered marriage""","""MALE""","""No""","""You are currently working""",7000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-18,1,0,0,1,7000.0
34919.0,2014.0,"""January""",19.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",48.5,"""10 and more grades of school &…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",4500.0,null,null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""","""no""",2014-01-19,1,0,0,1,4500.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59407.0,2024.0,"""November""",27.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",8.0,null,null,"""FEMALE""","""No""",null,null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2024-11-27,0,0,0,0,0.0
59408.0,2024.0,"""October""",9.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",73.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",31000.0,null,31000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",2024-10-09,0,0,0,0,0.0
59409.0,

In [158]:
panel  = panel.filter(pl.col('net_job') < 1_000_000)
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,date,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,i32,i32,i32,i32,f64
34896.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",66.0,"""Institute, University, Academy…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",44000.0,null,14000.0,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,1,0,0,1,30000.0
34897.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",62.5,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14000.0,null,13200.0,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,0,0,0,0,800.0
34907.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",54.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""Yes""","""You are not working""",7000.0,null,7000.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""",2014-01-18,0,1,0,0,0.0
34908.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",58.5,"""Technical, medical, musis etc …","""In a registered marriage""","""MALE""","""No""","""You are currently working""",7000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-18,1,0,0,1,7000.0
34919.0,2014.0,"""January""",19.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",48.5,"""10 and more grades of school &…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",4500.0,null,null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""","""no""",2014-01-19,1,0,0,1,4500.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59407.0,2024.0,"""November""",27.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",8.0,null,null,"""FEMALE""","""No""",null,null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2024-11-27,0,0,0,0,0.0
59408.0,2024.0,"""October""",9.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",73.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",31000.0,null,31000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",2024-10-09,0,0,0,0,0.0
59409.0,

In [159]:
(
    panel
        .filter((pl.col('is_employed_no_disablity') == 1)|(pl.col('is_employed_has_disablity') == 1))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'has_disablity']).agg(pl.col('net_job').mean())
)#.plot.line(x='year', y='net_job', color='has_disablity')

year,has_disablity,net_job
date,i32,f64
2017-01-01,0,27265.28691
2023-01-01,0,44485.484193
2020-01-01,1,25589.862595
2016-01-01,0,25519.360187
2022-01-01,1,32208.307086
…,…,…
2017-01-01,1,21718.952857
2023-01-01,1,35548.351724
2016-01-01,1,19265.172575


In [160]:
panel = panel.with_columns(
    working_pop = pl.when((pl.col('is_employed_no_disablity') == 1)|(pl.col('is_employed_has_disablity') == 1)).then(1).otherwise(0)
)
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,date,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job,working_pop
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,i32,i32,i32,i32,f64,i32
34896.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",66.0,"""Institute, University, Academy…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",44000.0,null,14000.0,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,1,0,0,1,30000.0,1
34897.0,2014.0,"""January""",5.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",62.5,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14000.0,null,13200.0,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-05,0,0,0,0,800.0,0
34907.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",54.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""Yes""","""You are not working""",7000.0,null,7000.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Third group""",2014-01-18,0,1,0,0,0.0,0
34908.0,2014.0,"""January""",18.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",58.5,"""Technical, medical, musis etc …","""In a registered marriage""","""MALE""","""No""","""You are currently working""",7000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2014-01-18,1,0,0,1,7000.0,1
34919.0,2014.0,"""January""",19.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",48.5,"""10 and more grades of school &…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",4500.0,null,null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""","""no""",2014-01-19,1,0,0,1,4500.0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59407.0,2024.0,"""November""",27.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",8.0,null,null,"""FEMALE""","""No""",null,null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2024-11-27,0,0,0,0,0.0,0
59408.0,2024.0,"""October""",9.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",73.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",31000.0,null,31000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",2

In [161]:
panel_working_pop_2019 = panel.filter(
    #pl.col('date').is_between(pl.date(2019, 1, 1), pl.date(2019, 12, 31))
    pl.col('year') == 2019
)
panel_working_pop_2019

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,date,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job,working_pop
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,i32,i32,i32,i32,f64,i32
1.0,2019.0,"""October""",11.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",46.0,"""Institute, University, Academy…","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",37000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2019-10-11,1,0,0,1,37000.0,1
7.0,2019.0,"""October""",9.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",75.0,"""7-9 grades of school [unfinish…","""Never married""","""FEMALE""","""Yes""","""You are not working""",16374.0,null,15424.0,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""Yes""","""No""","""No""","""Third group""","""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""Yes""","""No""","""No""","""Third group""",2019-10-09,0,1,0,0,950.0,0
9.0,2019.0,"""November""",13.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",73.0,"""Technical, medical, musis etc …","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",16000.0,null,16000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",2019-11-13,0,0,0,0,0.0,0
11292.0,2019.0,"""October""",15.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",77.5,"""Technical, medical, musis etc …","""Widower or widow""","""FEMALE""","""Yes""","""You are not working""",14689.0,null,12989.0,"""Yes""","""No""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Second group""","""Yes""","""No""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Second group""",2019-10-15,0,1,0,0,1700.0,0
3.0,2019.0,"""October""",9.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",64.5,"""10 and more grades of school &…","""Widower or widow""","""FEMALE""","""Yes""","""You are currently working""",38100.0,null,17500.0,"""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""No""","""Third group""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""No""","""Third group""",2019-10-09,1,1,1,0,20600.0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59409.0,2019.0,"""January""",23.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",48.5,"""10 and more grades of school &…","""Widower or widow""","""MALE""","""No""","""You are not working""",0.0,null,null,"""Yes""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""",null,"""Yes""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""no""",2019-01-23,0,0,0,0,0.0,0
59410.0,2019.0,"""January""",28.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""town""",70.5,"""9 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",46000.0,null,17000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No"""

In [162]:
panel_working_pop_2020 = panel.filter(
    #pl.col('date').is_between(pl.date(2020, 1, 1), pl.date(2020, 12, 31))
    pl.col('year') == 2020
)
panel_working_pop_2020

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,date,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job,working_pop
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,i32,i32,i32,i32,f64,i32
1.0,2020.0,"""October""",13.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",47.0,"""Institute, University, Academy…","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",37000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2020-10-13,1,0,0,1,37000.0,1
7.0,2020.0,"""October""",8.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",76.0,"""7-9 grades of school [unfinish…","""Never married""","""FEMALE""","""Yes""","""You are not working""",17376.0,null,17376.0,"""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""Yes""","""Yes""","""No""","""Second group""","""Yes""","""No""","""No""","""No""","""Yes""","""Yes""","""Yes""","""Yes""","""No""","""Second group""",2020-10-08,0,1,0,0,0.0,0
9.0,2020.0,"""October""",28.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",74.0,"""Technical, medical, musis etc …","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",22000.0,null,22000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2020-10-28,0,0,0,0,0.0,0
11292.0,2020.0,"""November""",11.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",78.5,"""Technical, medical, musis etc …","""Widower or widow""","""FEMALE""","""Yes""","""You are not working""",32550.0,null,16750.0,"""Yes""","""No""","""Yes""","""No""","""No""","""Yes""","""Yes""","""Yes""","""Yes""","""Second group""","""Yes""","""No""","""Yes""","""No""","""No""","""Yes""","""Yes""","""Yes""","""Yes""","""Second group""",2020-11-11,0,1,0,0,15800.0,0
3.0,2020.0,"""October""",9.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",65.5,"""10 and more grades of school &…","""Widower or widow""","""FEMALE""","""Yes""","""You are not working""",19700.0,null,19000.0,"""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""No""","""Third group""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""No""","""Third group""",2020-10-09,0,1,0,0,700.0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59408.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",69.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",11660.0,null,10660.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2020-09-12,0,0,0,0,1000.0,0
59409.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",49.0,"""10 and more grades of school &…","""Divorsed and not remarried""","""MALE""","""No""","""You are not working""",0.0,null,null,"""No""","""No""","""No""","""No""","""No""",null,"""No""",null,"""No""",null,

In [163]:
common_people = panel_working_pop_2019[['idind']]#.join(panel_working_pop_2020[['idind']], how='inner', on='idind').sort('idind').unique()
common_people

idind
f64
1.0
7.0
9.0
11292.0
3.0
…
59409.0
59410.0
59411.0


In [164]:
people_working_at_least_at_2020_2019 = panel.filter(
    ((pl.col('working_pop') == 1)&(pl.col('year') == 2019)) | ((pl.col('working_pop') == 1)&(pl.col('year') == 2020))
).sort('idind')['idind'].unique().to_frame()
people_working_at_least_at_2020_2019

idind
f64
1.0
3.0
6.0
29.0
30.0
…
60467.0
60469.0
60470.0


In [165]:
common_people = common_people[['idind']].join(people_working_at_least_at_2020_2019[['idind']], how='inner', on='idind').sort('idind').unique()
common_people

idind
f64
1.0
3.0
6.0
29.0
30.0
…
59395.0
59397.0
59398.0


In [166]:
panel_common_2019_2020 = panel.filter(
    pl.col('idind').is_in(common_people['idind'].implode())
).sort('idind')
panel_common_2019_2020

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,date,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job,working_pop
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,i32,i32,i32,i32,f64,i32
1.0,2015.0,"""October""",25.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",42.0,"""Institute, University, Academy…","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",40000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2015-10-25,1,0,0,1,40000.0,1
1.0,2016.0,"""November""",1.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",43.0,"""Institute, University, Academy…","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",47900.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2016-11-01,1,0,0,1,47900.0,1
1.0,2017.0,"""September""",20.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",44.0,"""Institute, University, Academy…","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",4000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2017-09-20,1,0,0,1,4000.0,1
1.0,2018.0,"""October""",17.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",45.0,"""Institute, University, Academy…","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",47000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2018-10-17,1,0,0,1,47000.0,1
1.0,2019.0,"""October""",11.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",46.0,"""Institute, University, Academy…","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",37000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2019-10-11,1,0,0,1,37000.0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59400.0,2022.0,"""January""",11.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",63.0,"""10 and more grades of school &…","""Widower or widow""","""FEMALE""","""No""","""You are currently working""",32500.0,null,17500.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",2022-01-11,1,0,0,1,15000.0,1
59400.0,2023.0,"""January""",21.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",64.0,"""10 and more grades of school &…","""Widower or widow""","""FEMALE""","""No""","""You are currently working""",45000.0,null,26000.0,"""Yes""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""Yes""","""No""",""

In [167]:
panel_common_2019_2020.filter(
    pl.col('is_employed_no_disablity') == 1
).group_by('year').len().plot.line(x='year', y='len')

alt.Chart(...)

In [168]:
panel_common_2019_2020.filter(
    pl.col('is_employed_has_disablity') == 1
).group_by('year').len().plot.line(x='year', y='len')

alt.Chart(...)

In [169]:
(
    panel_common_2019_2020
        #.filter((pl.col('is_employed_no_disablity') == 1)|(pl.col('is_employed_has_disablity') == 1))
        .filter(pl.col('h7_2').is_in(['October', 'November', 'December']))
        .filter(pl.col('working_pop') == 1)
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='has_disablity')

alt.Chart(...)

In [170]:
working_data = (
    panel_common_2019_2020

        .join(education_levels, on='educ', how='left', validate='m:1')
        .with_columns(pl.col('educ_level').fill_null('school_or_less'))
        .join(marital_status, on='marst', how='left', validate='m:1')
        .join(period_relevance, on='year', how='left', validate='m:1')
        .join(is_town, on='status', how='left', validate='m:1')
        .with_columns(pl.col('is_married').fill_null(0))
        .with_columns(
            is_female = pl.when(pl.col('h5') == 'FEMALE').then(1).otherwise(0),
        ).with_columns(
            pl.when(pl.col('chronic_heart') == 'Yes').then(1).otherwise(0).alias('chronic_heart'),
            pl.when(pl.col('chronic_lung') == 'Yes').then(1).otherwise(0).alias('chronic_lung'),
            pl.when(pl.col('chronic_liver') == 'Yes').then(1).otherwise(0).alias('chronic_liver'),
            pl.when(pl.col('chronic_kidney') == 'Yes').then(1).otherwise(0).alias('chronic_kidney'),
            pl.when(pl.col('chronic_stomach') == 'Yes').then(1).otherwise(0).alias('chronic_stomach'),
            pl.when(pl.col('chronic_spinal') == 'Yes').then(1).otherwise(0).alias('chronic_spinal'),
            pl.when(pl.col('disease_neuro') == 'Yes').then(1).otherwise(0).alias('disease_neuro'),
            pl.when(pl.col('disease_eye') == 'Yes').then(1).otherwise(0).alias('disease_eye'),
            pl.when(pl.col('allegies') == 'Yes').then(1).otherwise(0).alias('allegies')
        )
        #.filter(pl.col('idind').is_in(common_people))
).drop(cols_disease)
working_data = (
    working_data
        .rename({'status':'habitat'}).to_dummies('habitat')
        .to_dummies('educ_level')
        .to_dummies('disability_class')
        .drop('disability_class_no')
        .rename({'disability_class_First group':'disability_class_first_group', 'disability_class_Second group':'disability_class_second_group', 'disability_class_Third group':'disability_class_third_group'})
)
working_data

idind,year,h7_2,h7_1,origsm,region,psu,habitat_PGT,habitat_oblastnoy center,habitat_rural,habitat_town,age,educ,marst,h5,m20_7,j1,j60,j40,j363,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class_first_group,disability_class_second_group,disability_class_third_group,date,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job,working_pop,educ_level_common,educ_level_higher,educ_level_school_or_less,educ_level_university,is_married,period_relevance,is_town,is_female
f64,f64,str,f64,str,str,str,u8,u8,u8,u8,f64,str,str,str,str,str,f64,f64,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,u8,u8,u8,date,i32,i32,i32,i32,f64,i32,u8,u8,u8,u8,i64,i64,i64,i32
1.0,2015.0,"""October""",25.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…",1,0,0,0,42.0,"""Institute, University, Academy…","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",40000.0,null,null,0,0,0,0,0,0,0,0,0,0,0,0,2015-10-25,1,0,0,1,40000.0,1,0,0,0,1,0,-4,1,1
1.0,2016.0,"""November""",1.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…",1,0,0,0,43.0,"""Institute, University, Academy…","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",47900.0,null,null,0,0,0,0,0,0,0,0,0,0,0,0,2016-11-01,1,0,0,1,47900.0,1,0,0,0,1,0,-3,1,1
1.0,2017.0,"""September""",20.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…",1,0,0,0,44.0,"""Institute, University, Academy…","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",4000.0,null,null,0,0,0,0,0,0,0,0,0,0,0,0,2017-09-20,1,0,0,1,4000.0,1,0,0,0,1,0,-2,1,1
1.0,2018.0,"""October""",17.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…",1,0,0,0,45.0,"""Institute, University, Academy…","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",47000.0,null,null,0,0,0,0,0,0,0,0,0,0,0,0,2018-10-17,1,0,0,1,47000.0,1,0,0,0,1,0,-1,1,1
1.0,2019.0,"""October""",11.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…",1,0,0,0,46.0,"""Institute, University, Academy…","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",37000.0,null,null,0,0,0,0,0,0,0,0,0,0,0,0,2019-10-11,1,0,0,1,37000.0,1,0,0,0,1,0,0,1,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59400.0,2022.0,"""January""",11.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""",1,0,0,0,63.0,"""10 and more grades of school &…","""Widower or widow""","""FEMALE""","""No""","""You are currently working""",32500.0,null,17500.0,0,0,0,0,0,0,0,0,0,0,0,0,2022-01-11,1,0,0,1,15000.0,1,0,1,0,0,0,3,1,1
59400.0,2023.0,"""January""",21.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""",1,0,0,0,64.0,"""10 and more grades of school &…","""Widower or widow""","""FEMALE""","""No""","""You are currently working""",45000.0,null,26000.0,1,0,0,0,0,0,0,0,0,0,0,0,2023-01-21,1,0,0,1,19000.0,1,0,1,0,0,0,4,1,1
59400.0,2024.0,"""November""",15.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""",1,0,0,0,64.5,"""10 and more grades of school &…","""Widower or widow""","""FEMALE""","""No""","""You are currently working""",41000.0,null,23000.0,0,0,0,0,0,0,0,0,0,0,0,0,2024-11-15,1,0,0,1,18000.0,1,0,1,0,0,0,5,1,1


In [171]:
working_data = (
    working_data
        .filter(pl.col('h7_2').is_in(['October', 'November', 'December']))
    .group_by(['idind', 'year']).agg(
        pl.col('is_employed').mean(), pl.col('has_disablity').mean(), pl.col('j60').mean(), pl.col('net_job').mean(), pl.col('is_employed_has_disablity').last(), pl.col('age').last(), pl.col('is_female').last(), pl.col('is_town').last(), pl.col('educ_level_school_or_less').last(), pl.col('educ_level_common').last(), pl.col('educ_level_higher').last(), pl.col('educ_level_university').last(), pl.col('is_married').last(), pl.col('chronic_heart').last(), pl.col('chronic_lung').last(), pl.col('chronic_liver').last(), pl.col('chronic_kidney').last(), pl.col('chronic_stomach').last(), pl.col('chronic_spinal').last(), pl.col('disease_neuro').last(), pl.col('disease_eye').last(), pl.col('allegies').last(), pl.col('disability_class_first_group').last(), pl.col('disability_class_second_group').last(), pl.col('disability_class_third_group').last(), pl.col('period_relevance').last()
    )
    .sort(['idind', 'year'])
)
working_data

idind,year,is_employed,has_disablity,j60,net_job,is_employed_has_disablity,age,is_female,is_town,educ_level_school_or_less,educ_level_common,educ_level_higher,educ_level_university,is_married,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class_first_group,disability_class_second_group,disability_class_third_group,period_relevance
f64,f64,f64,f64,f64,f64,i32,f64,i32,i64,u8,u8,u8,u8,i64,i32,i32,i32,i32,i32,i32,i32,i32,i32,u8,u8,u8,i64
1.0,2015.0,1.0,0.0,40000.0,40000.0,0,42.0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,-4
1.0,2016.0,1.0,0.0,47900.0,47900.0,0,43.0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,-3
1.0,2018.0,1.0,0.0,47000.0,47000.0,0,45.0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,-1
1.0,2019.0,1.0,0.0,37000.0,37000.0,0,46.0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.0,2020.0,1.0,0.0,37000.0,37000.0,0,47.0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59398.0,2021.0,1.0,0.0,29000.0,29000.0,0,37.0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,2
59398.0,2024.0,1.0,0.0,70000.0,70000.0,0,40.0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,5
59400.0,2021.0,1.0,0.0,22400.0,12000.0,0,61.5,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2


In [172]:
# working_data = working_data.filter(
#     pl.col('idind').is_in(
#         working_data.filter(pl.col('year').is_in([2019, 2020]))['idind'].unique().implode()
#     )
# )
# working_data

In [173]:
working_data.filter(pl.col('is_employed_has_disablity') == 1).plot.boxplot(y='net_job').properties(width=100, height=350)

alt.Chart(...)

In [174]:
working_data.filter(pl.col('is_employed_has_disablity') == 0).plot.boxplot(y='net_job').properties(width=100, height=350)

alt.Chart(...)

In [175]:
ids_to_exclude_0 = working_data.filter(pl.col('is_employed_has_disablity') == 0).filter(pl.col('net_job') > 500_000)['idind'].to_list()
#ids_to_exclude_0 = []
ids_to_exclude_0

[4326.0,
 14137.0,
 14137.0,
 14137.0,
 14137.0,
 14137.0,
 26759.0,
 35638.0,
 36089.0,
 36635.0,
 39478.0,
 49130.0,
 50863.0,
 51399.0,
 51399.0,
 51497.0,
 52451.0]

In [176]:
ids_to_exclude_1 = working_data.filter(pl.col('is_employed_has_disablity') == 1).filter(pl.col('net_job') > 120000)['idind'].to_list()
ids_to_exclude_1 = [x for x in ids_to_exclude_1 if x not in [7399, 40749]]
ids_to_exclude_1

[33674.0, 58780.0, 58791.0, 59156.0, 59156.0]

In [182]:
(
    working_data
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'is_employed_has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='is_employed_has_disablity')

alt.Chart(...)

In [178]:
panel2019_2020 = (
    working_data
        # .filter(pl.col())
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .filter(pl.col('year').is_in([2019, 2020]))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .filter(pl.col('idind').is_duplicated())
).to_pandas().set_index(['idind', 'year'])
panel2019_2020

is_employed  has_disablity       j60   net_job  \
idind   year                                                         
1.0     2019-01-01          1.0            0.0   37000.0   37000.0   
        2020-01-01          1.0            0.0   37000.0   37000.0   
3.0     2019-01-01          1.0            1.0   38100.0   20600.0   
        2020-01-01          0.0            1.0   19700.0     700.0   
6.0     2019-01-01          1.0            0.0   30500.0   30500.0   
...                         ...            ...       ...       ...   
59268.0 2020-01-01          1.0            0.0   26300.0   26300.0   
59271.0 2019-01-01          1.0            0.0   40000.0   40000.0   
        2020-01-01          1.0            0.0   36000.0   36000.0   
59274.0 2019-01-01          0.0            0.0   18980.0       0.0   
        2020-01-01          1.0            0.0  118996.0  100000.0   

                    is_employed_has_disablity   age  is_female  is_town  \
idind   year                                                              
1.0     2019-01-01                          0  46.0          1        1   
        2020-01-01                          0  47.0          1        1   
3.0     2019-01-01                          1  64.5          1        1   
        2020-01-01                          0  65.5          1        1   
6.0     2019-01-01                          0  33.5          0        1   
...                                       ...   ...        ...      ...   
59268.0 2020-01-01                          0  27.5          1        1   
59271.0 2019-01-01                          0  34.5          0        1   
        2020-01-01                          0  35.5          0        1   
59274.0 2019-01-01                          0  59.5          1        1   
        2020-01-01                          0  61.0          1        1   

                    educ_level_school_or_less  educ_level_common  ...  \
idind   year                                                      ...   
1.0     2019-01-01                          0                  0  ...   
        2020-01-01                          0                  0  ...   
3.0     2019-01-01                          0                  0  ...   
        2020-01-01                          0                  0  ...   
6.0     2019-01-01                          0                  1  ...   
...                                       ...                ...  ...   
59268.0 2020-01-01                          0                  0  ...   
59271.0 2019-01-01                          0                  0  ...   
        2020-01-01                          0                  0  ...   
59274.0 2019-01-01                          0                  0  ...   
        2020-01-01                          0                  0  ...   

                    chronic_kidney  chronic_stomach  chronic_spinal  \
idind   year                                                          
1.0     2019-01-01               0                0               0   
        2020-01-01               0                0               0   
3.0     2019-01-01               0                1               0   
        2020-01-01               0                1               0   
6.0     2019-01-01               0                0               0   
...                            ...              ...             ...   
59268.0 2020-01-01               0                0               0   
59271.0 2019-01-01               0                0               0   
        2020-01-01               0                0               0   
59274.0 2019-01-01               0                0               0   
        2020-01-01               0                0               0   

                    disease_neuro  disease_eye  allegies  \
idind   year                                               
1.0     2019-01-01              0            0         0   
        2020-01-01              0            0         0   
3.0     2019-01-01 

In [183]:
(
    working_data
        # .filter(pl.col())
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .filter(pl.col('year').is_in([2019, 2020]))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .filter(pl.col('idind').is_duplicated())
)

idind,year,is_employed,has_disablity,j60,net_job,is_employed_has_disablity,age,is_female,is_town,educ_level_school_or_less,educ_level_common,educ_level_higher,educ_level_university,is_married,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class_first_group,disability_class_second_group,disability_class_third_group,period_relevance
f64,date,f64,f64,f64,f64,i32,f64,i32,i64,u8,u8,u8,u8,i64,i32,i32,i32,i32,i32,i32,i32,i32,i32,u8,u8,u8,i64
1.0,2019-01-01,1.0,0.0,37000.0,37000.0,0,46.0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.0,2020-01-01,1.0,0.0,37000.0,37000.0,0,47.0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3.0,2019-01-01,1.0,1.0,38100.0,20600.0,1,64.5,1,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0
3.0,2020-01-01,0.0,1.0,19700.0,700.0,0,65.5,1,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1
6.0,2019-01-01,1.0,0.0,30500.0,30500.0,0,33.5,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59268.0,2020-01-01,1.0,0.0,26300.0,26300.0,0,27.5,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1
59271.0,2019-01-01,1.0,0.0,40000.0,40000.0,0,34.5,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
59271.0,2020-01-01,1.0,0.0,36000.0,36000.0,0,35.5,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1


In [185]:
model1 = PanelOLS.from_formula('np.log(net_job + 1) ~ has_disablity + EntityEffects', data=panel2019_2020).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model1.summary)

                           PanelOLS Estimation Summary                           
Dep. Variable:     np.log(net_job + 1)   R-squared:                        0.0020
Estimator:                    PanelOLS   R-squared (Between):             -0.0097
No. Observations:                11934   R-squared (Within):               0.0020
Date:                 Tue, Apr 21 2026   R-squared (Overall):             -0.0094
Time:                         12:52:22   Log-likelihood                -2.236e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      12.159
Entities:                         5967   P-value                           0.0005
Avg Obs:                        2.0000   Distribution:                  F(1,5966)
Min Obs:                        2.0000                                           
Max Obs:                        2.0000   F-statistic (robust):             4.9155
                

In [186]:
model2 = PanelOLS.from_formula('net_job ~ has_disablity + EntityEffects + TimeEffects', data=panel2019_2020).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model2.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                net_job   R-squared:                        0.0005
Estimator:                   PanelOLS   R-squared (Between):             -0.0063
No. Observations:               11934   R-squared (Within):               0.0005
Date:                Tue, Apr 21 2026   R-squared (Overall):             -0.0058
Time:                        13:04:05   Log-likelihood                -1.276e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      2.9892
Entities:                        5967   P-value                           0.0839
Avg Obs:                       2.0000   Distribution:                  F(1,5965)
Min Obs:                       2.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             2.8044
                            